In [36]:
import math
from collections import Counter, defaultdict


def makeLower(text):
    return text.lower().strip()

def load_file(path):
    with open(path, encoding="utf8") as f:
        return makeLower(f.read())
    
def stats(text, alphabet):
    counter = Counter()
    for c in text:
        if c in alphabet:
            counter[c] += 1    
        else:
            counter["<unk>"] += 1
        total = sum(counter.values())
    return counter, total

# use laplace smoothing 
def Laplace(chara, counts, total, V):
    #           Ci + 1                     N + V
    return (counts.get(chara, 0) + 1) / (total + V)

def logprob(sentence, counts, total, alphabet):
    V = len(alphabet) + 1  # add one for <unk>
    logp = 0.0
    for c in makeLower(sentence):
        if c in alphabet:                                   # ignore OOV characters
            logp += math.log(Laplace(c, counts, total, V))  # log the sum of the probabilities ]
        else:
            logp += math.log(Laplace("<unk>", counts, total, V))
    return logp

def predict_language(sentence, models, alphabet):
    scores = {}
    for lang, (counts, total) in models.items():
        scores[lang] = logprob(sentence, counts, total, alphabet)
    return max(scores, key=scores.get), scores

alphabet = list("abcdefghijklmnopqrstuvwxyz") + ['à', 'á', 'â', 'ä','ç', 'é', 'è', 'ê', 'ë', 'î', 
                                                  'ï', 'ô', 'ö','ù', 'û', 'ü', 'ÿ', 'ò', 'ó', 
                                                  'ì', 'í', 'á', 'ú', 'ä', 'œ', 'æ', "'", 
                                                  '’', '«', '»', '-', ' ', 'ä', 'ö','“', '”', '"'] 

#load data to train 
en_text = load_file("../Data/Input/LangId.train.English")
fr_text  = load_file("../Data/Input/LangId.train.French")
it_text  = load_file("../Data/Input/LangId.train.Italian")

en_counts, en_total   = stats(en_text, alphabet)
fr_counts, fr_total   = stats(fr_text, alphabet)
it_counts, it_total   = stats(it_text, alphabet)

models = {
    "English": (en_counts, en_total),
    "French":  (fr_counts, fr_total),
    "Italian": (it_counts, it_total)
}




with open("../Data/Validation/LangId.test", encoding="utf8") as f:
    test_lines = f.readlines()

    output_lines = []

    for i, line in enumerate(test_lines, 1):
        # get the ID for numbering the lines from enumerate 
        line_id = str(i)
        
        # get the sentence  
        sentence = line.strip() 
        pred = predict_language(sentence, models, alphabet)[0]
        
        # output string shpuld be in form "ID Language\n"
        output_lines.append(f"{line_id} {pred}\n")

with open("../Data/Output/letterLangId.out", "w", encoding="utf8") as f:
    f.writelines(output_lines)

with open("../Data/Validation/labels.sol", encoding="utf8") as f:
    sol = f.readlines()

    correct = 0
    total = len(sol)

    for mine, sol in zip(output_lines, sol):                # parallel comparison of output with solution
        if mine.strip() == sol.strip():
            correct += 1

    print(f"Correct: {correct}/{total}")
    print(f"Accuracy: {correct/total:.2%}")



Correct: 290/300
Accuracy: 96.67%
